In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

In [ ]:
dataset = fetch_openml('car')
df = dataset.data
df['target'] = dataset.target

In [ ]:
df

In [ ]:
def chi_square_scratch(df, attribute, target):
  attr_values = df[attribute].unique()
  target_values=df[target].unique()
  chi2=0
  for attr_value in attr_values:
    for target_value in target_values:
      observed_freq = len(df[(df[attribute]==attr_value)&(df[target]==target_value)])
      attr_prob = len(df[df[attribute]==attr_value])/len(df)
      target_prob = len(df[(df[target]==target_value)])/len(df)
      expected_freq = attr_prob*target_prob*len(df)
      chi2+=(observed_freq-expected_freq)**2/expected_freq
  return chi2

In [ ]:
chi_square_scratch(df,'buying','target')

In [ ]:
def selectkbest_chi2_scratch(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  chi2_score={}
  for attribute in attributes:
    chi2_score[attribute] = chi_square_scratch(df,attribute,target)
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_chi2_score[:k]]
  return selected_features,sorted_chi2_score

In [ ]:
selectkbest_chi2_scratch(df, 3)

In [ ]:
from scipy.stats import chi2_contingency
table = pd.crosstab(df['buying'],df['target'])
chi2_contingency(table)

In [ ]:
def SelectKbest_chi2_inbuilt(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  chi2_score={}
  for attribute in attributes:
    table = pd.crosstab(df[attribute],df[target])
    chi2,_,_,_=chi2_contingency(table)
    chi2_score[attribute]=chi2
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_chi2_score[:k]]
  return selected_features,sorted_chi2_score


In [ ]:
SelectKbest_chi2_inbuilt(df,3)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for column in df.columns:
  df[column] = le.fit_transform(df[column])
from sklearn.feature_selection import SelectKBest,chi2
test = SelectKBest(chi2, k=3)
test.fit(df.iloc[:,:-1],df.iloc[:,-1])
test.scores_

In [ ]:
df.columns[:-1][test.get_support()]

In [ ]:
def find_entropy(df):
  target = df.columns[-1]
  target_values = df[target].unique()
  entropy = 0
  for target_value in target_values:
    prob = len(df[df[target]==target_value])/len(df)
    entropy+=-prob*np.log(prob)
  return entropy

In [ ]:
find_entropy(df)

In [ ]:
def find_avg_info_entropy(df,attribute):
  attr_values = df[attribute].unique()
  target = df.columns[-1]
  target_values= df[target].unique()
  avg_info_entropy = 0
  for attr_value in attr_values:
    entropy_subsample = 0
    for target_value in target_values:
      num=len(df[(df[attribute]==attr_value)&(df[target]==target_value)])
      den=len(df[df[attribute]==attr_value])
      prob = num/den
      entropy_subsample=-prob*np.log(prob+1e-7)
    weight=len(df[df[attribute]==attr_value])/len(df)
    avg_info_entropy+=weight*entropy_subsample
  return avg_info_entropy

In [ ]:
find_avg_info_entropy(df,'buying')

In [ ]:
def selectkbest_IG_scratch(df,k):
  attributes = df.columns[:-1]
  IG_score={}
  for attribute in attributes:
    IG_score[attribute] = find_entropy(df)-find_avg_info_entropy(df,attribute)
  sorted_IG_score = sorted(IG_score.items(),key=lambda x:x[1],reverse=True)
  selected_features=[f for f,_ in sorted_IG_score[:k]]
  return selected_features,sorted_IG_score

In [ ]:
selectkbest_IG_scratch(df,3)

In [ ]:
from sklearn.feature_selection import SelectKBest,mutual_info_classif
test1 = SelectKBest(mutual_info_classif,k=3)
test1.fit(df.iloc[:,:-1],df.iloc[:,-1])
test1.scores_


In [ ]:
df.columns[:-1][test1.get_support()]

In [ ]:
from sklearn.metrics import mutual_info_score
mutual_info_score(df['buying'],df['target'])

In [ ]:
find_entropy(df)-find_avg_info_entropy(df,'buying')